# 03 · Extract the corpus and test set (the one big BigQuery query)

Scope chosen in notebook 02: **granted US patents filed 2012–2021 whose main CPC code is G06N** (about 17,600 families), plus recent G06N documents whose **examiner citations** point into that corpus (the test set).

1. Selects one granted patent per family for the corpus
2. Selects up to 1,000 test cases: documents with a **2022 or later priority date** and at least 2 examiner citations into the corpus
3. Extracts title, abstract, claims and applicant for all of them in **one query** (about 375 GB of the free 1,000 GB monthly quota, capped at 420 GB)
4. Saves everything to Google Drive

BigQuery charges for whole columns, so this costs the same whether we take 20,000 or 45,000 documents. It only needs to run once.

In [ ]:
PROJECT_ID = "patent-search-515813"
TABLE = "patents-public-data.patents.publications"
DRIVE_DIR = "/content/drive/MyDrive/patent-search"

from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")

import pandas as pd
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

meta = pd.read_parquet(f"{DRIVE_DIR}/g06n_us_meta.parquet")
cit = pd.read_parquet(f"{DRIVE_DIR}/g06n_us_citations.parquet")
meta["year"] = meta["filing_date"] // 10000
meta["is_grant"] = meta["kind_code"].str.startswith("B")
meta["first_cpc"] = meta["first_cpc"].fillna("")
print(f"Loaded {len(meta):,} documents and {len(cit):,} citations from Drive")

## 1 · Corpus: one granted patent per family

In [ ]:
corpus = meta[
    meta.is_grant
    & meta.year.between(2012, 2021)
    & meta.first_cpc.str.startswith("G06N")
].sort_values("grant_date").drop_duplicates("family_id", keep="last")   # latest grant per family

corpus_families = set(corpus["family_id"])
family_to_corpus_doc = dict(zip(corpus["family_id"], corpus["publication_number"]))
print(f"Corpus: {len(corpus):,} granted patents, one per family")
print(corpus["year"].value_counts().sort_index().to_string())

## 2 · Test set: recent documents and the corpus patents their examiner cited

In [ ]:
family_of = dict(zip(meta["publication_number"], meta["family_id"]))
exam = cit[cit["category"].fillna("").str.contains("SEA")].copy()
exam["cited_family"] = exam["cited"].map(family_of)
exam = exam[exam["cited_family"].isin(corpus_families)]

recent = meta[(meta["priority_date"] >= 20220101) & ~meta["family_id"].isin(corpus_families)]
exam = exam[exam["publication_number"].isin(set(recent["publication_number"]))]

per_doc = exam.groupby("publication_number")["cited_family"].apply(lambda s: sorted(set(s)))
per_doc = per_doc[per_doc.str.len() >= 2]

# One test document per family: the one with the most examiner citations into the corpus
cands = per_doc.to_frame("cited_families").reset_index()
cands["family_id"] = cands["publication_number"].map(family_of)
cands["n_cited"] = cands["cited_families"].str.len()
cands = cands.sort_values("n_cited", ascending=False).drop_duplicates("family_id")
print(f"Candidate test cases (2022+ priority, >=2 examiner citations into the corpus): {len(cands):,}")

testset = cands.sample(n=min(1000, len(cands)), random_state=42)
testset["relevant"] = testset["cited_families"].apply(lambda fams: [family_to_corpus_doc[f] for f in fams])
testset = testset[["publication_number", "family_id", "relevant"]]
print(f"Test cases kept: {len(testset):,}")
print("Examiner-cited corpus patents per test case:")
print(testset["relevant"].str.len().describe().round(1).to_string())

## 3 · Extract the text (one query)

In [ ]:
ids = sorted(set(corpus["publication_number"]) | set(testset["publication_number"]))
print(f"Documents to extract: {len(ids):,}")

SQL_TEXT = f"""
SELECT
  publication_number,
  (SELECT t.text FROM UNNEST(title_localized) t WHERE t.language = 'en' LIMIT 1) AS title,
  (SELECT t.text FROM UNNEST(abstract_localized) t WHERE t.language = 'en' LIMIT 1) AS abstract,
  (SELECT t.text FROM UNNEST(claims_localized) t WHERE t.language = 'en' LIMIT 1) AS claims,
  ARRAY(SELECT a.name FROM UNNEST(assignee_harmonized) a) AS assignees
FROM `{TABLE}`
WHERE publication_number IN UNNEST(@ids)
"""
params = [bigquery.ArrayQueryParameter("ids", "STRING", ids)]

dry = client.query(SQL_TEXT, job_config=bigquery.QueryJobConfig(
    dry_run=True, use_query_cache=False, query_parameters=params))
gb = dry.total_bytes_processed / 1e9
print(f"Estimated scan: {gb:.1f} GB")

MAX_GB = 420
assert gb <= MAX_GB, "Above the limit: stopping. Nothing was charged."
job = client.query(SQL_TEXT, job_config=bigquery.QueryJobConfig(
    query_parameters=params, maximum_bytes_billed=int(MAX_GB * 1e9)))
text = job.to_dataframe()
print(f"Extracted {len(text):,} documents")

## 4 · Check and save

In [ ]:
text["assignees"] = text["assignees"].apply(lambda a: "; ".join(a) if a is not None else "")
for col in ["title", "abstract", "claims"]:
    missing = (text[col].fillna("").str.len() == 0).sum()
    words = text[col].fillna("").str.split().str.len()
    print(f"{col:9} missing: {missing:5,} | median words: {int(words.median()):,}")

keep_meta = ["publication_number", "family_id", "kind_code", "filing_date",
             "priority_date", "grant_date", "first_cpc", "g06n_codes"]
corpus_text = corpus[keep_meta].merge(text, on="publication_number", how="left")
corpus_text["g06n_codes"] = corpus_text["g06n_codes"].apply(list)
query_text = testset.merge(text, on="publication_number", how="left")
query_text["relevant"] = query_text["relevant"].apply(list)

corpus_text.to_parquet(f"{DRIVE_DIR}/corpus_text.parquet", index=False)
query_text.to_parquet(f"{DRIVE_DIR}/testset_text.parquet", index=False)
print(f"\nSaved corpus_text.parquet ({len(corpus_text):,} patents) and testset_text.parquet ({len(query_text):,} test cases)")

In [ ]:
# One example from each file
ex = corpus_text.iloc[0]
print("CORPUS:", ex.publication_number, "|", ex.title, "|", ex.assignees)
print("Abstract:", (ex.abstract or "")[:300], "...")
print("Claims start:", (ex.claims or "")[:300], "...")

q = query_text.iloc[0]
print("\nTEST CASE:", q.publication_number, "|", q.title)
print("Examiner cited in corpus:", q.relevant)